# Herramienta 03 - Recomendación personalizada de destinos

Este notebook implementa una herramienta de recomendación: prepara interacciones usuario-destino, evalúa Precision@K y Recall@K, y permite consultar recomendaciones explicadas para un usuario.


## 1. Configuración
Se usan pandas, numpy y similitud coseno. El notebook puede ejecutarse en Colab sin clonar el repositorio.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

SEED = 42
np.random.seed(SEED)
DATA_URL = 'https://raw.githubusercontent.com/AndresGuido9820/sistema-transporte-inteligente/main/data/processed/travel_interactions.csv'


## 2. Carga de interacciones
Cada fila representa una interacción usuario-destino con calificación o preferencia observada.


In [ ]:
def generar_interacciones(usuarios=60, destinos=14):
    nombres_destinos = [f'Destino_{i:02d}' for i in range(1, destinos + 1)]
    filas = []
    for u in range(1, usuarios + 1):
        preferidos = np.random.choice(nombres_destinos, size=4, replace=False)
        for destino in np.random.choice(nombres_destinos, size=np.random.randint(5, 10), replace=False):
            rating = np.random.randint(4, 6) if destino in preferidos else np.random.randint(1, 5)
            filas.append({'user_id': f'U{u:03d}', 'destination': destino, 'rating': rating})
    return pd.DataFrame(filas)

try:
    interacciones = pd.read_csv(DATA_URL)
    print('Datos cargados desde el repositorio:', interacciones.shape)
except Exception as exc:
    print('No se pudo cargar el CSV remoto. Se usará muestra generada.', exc)
    interacciones = generar_interacciones()

interacciones.head()


## 3. Análisis exploratorio
Se revisa la actividad de usuarios, popularidad de destinos y distribución de calificaciones.


In [ ]:
print('Usuarios:', interacciones['user_id'].nunique())
print('Destinos:', interacciones['destination'].nunique())

display(interacciones.groupby('destination')['rating'].agg(['count', 'mean']).sort_values('count', ascending=False).head(10).round(2))

plt.figure(figsize=(8, 3))
interacciones['rating'].value_counts().sort_index().plot(kind='bar')
plt.title('Distribución de calificaciones')
plt.xlabel('Rating')
plt.ylabel('Interacciones')
plt.grid(axis='y', alpha=0.25)
plt.show()


## 4. Matriz usuario-destino
La matriz representa el historial de cada usuario. Los valores faltantes se llenan con 0 para calcular similitud.


In [ ]:
matriz = interacciones.pivot_table(index='user_id', columns='destination', values='rating', fill_value=0)
print('Matriz:', matriz.shape)
display(matriz.head())


## 5. Evaluación leave-one-out
Se oculta un destino bien calificado por usuario y se comprueba si el recomendador logra recuperarlo entre los Top-K.


In [ ]:
def evaluar_recomendador(matriz, k=5):
    rng = np.random.default_rng(SEED)
    train = matriz.copy()
    ocultos = {}
    for usuario in matriz.index:
        positivos = list(matriz.columns[matriz.loc[usuario] >= 4])
        if len(positivos) >= 2:
            destino = rng.choice(positivos)
            train.loc[usuario, destino] = 0
            ocultos[usuario] = destino

    similitud = cosine_similarity(train)
    usuarios = list(train.index)
    destinos = list(train.columns)
    filas = []
    for idx, usuario in enumerate(usuarios):
        if usuario not in ocultos:
            continue
        scores = similitud[idx] @ train.values
        vistos = set(train.columns[train.loc[usuario] > 0])
        ranking = [dest for _, dest in sorted(zip(scores, destinos), reverse=True) if dest not in vistos][:k]
        hit = int(ocultos[usuario] in ranking)
        filas.append({'user_id': usuario, 'hidden_destination': ocultos[usuario], 'hit': hit, 'recommendations': ranking})
    resultados = pd.DataFrame(filas)
    metricas = {'Precision@K': resultados['hit'].mean() / k, 'Recall@K': resultados['hit'].mean()}
    return train, similitud, resultados, metricas

K = 5
train_matrix, similitud, eval_df, metricas = evaluar_recomendador(matriz, K)
print(metricas)
display(eval_df.head())


## 6. Motor de recomendación
La recomendación se genera a partir de usuarios similares. También se entrega una explicación simple para facilitar la demostración.


In [ ]:
def recomendar_usuario(usuario, matriz_entrenamiento, similitud, k=5):
    usuarios = list(matriz_entrenamiento.index)
    destinos = list(matriz_entrenamiento.columns)
    if usuario not in usuarios:
        return pd.DataFrame(columns=['rank', 'destination', 'score', 'reason'])

    idx = usuarios.index(usuario)
    scores = similitud[idx] @ matriz_entrenamiento.values
    vistos = set(matriz_entrenamiento.columns[matriz_entrenamiento.loc[usuario] > 0])
    vecinos = sorted([(s, u) for s, u in zip(similitud[idx], usuarios) if u != usuario], reverse=True)[:3]
    vecinos_ids = [u for _, u in vecinos]

    filas = []
    for score, destino in sorted(zip(scores, destinos), reverse=True):
        if destino in vistos:
            continue
        apoyo = [v for v in vecinos_ids if matriz_entrenamiento.loc[v, destino] > 0]
        razon = 'Preferido por usuarios con historial similar'
        if apoyo:
            razon = f'Coincide con usuarios similares: {", ".join(apoyo[:2])}'
        filas.append({'rank': len(filas) + 1, 'destination': destino, 'score': round(float(score), 3), 'reason': razon})
        if len(filas) == k:
            break
    return pd.DataFrame(filas)


## 7. Herramienta de recomendación
Seleccione un usuario y el número de recomendaciones. La tabla muestra destinos sugeridos, puntaje y justificación.


In [ ]:
#@title Parámetros de la herramienta
usuario_seleccionado = 'U001' #@param {type:'string'}
numero_recomendaciones = 5 #@param {type:'integer'}

if usuario_seleccionado not in train_matrix.index:
    usuario_seleccionado = train_matrix.index[0]
    print('Usuario no encontrado. Se usará:', usuario_seleccionado)

print('Historial del usuario')
display(interacciones[interacciones['user_id'] == usuario_seleccionado].sort_values('rating', ascending=False))

recomendaciones = recomendar_usuario(usuario_seleccionado, train_matrix, similitud, numero_recomendaciones)
print('Recomendaciones generadas')
display(recomendaciones)

plt.figure(figsize=(8, 3))
plt.bar(recomendaciones['destination'], recomendaciones['score'])
plt.title(f'Top recomendaciones para {usuario_seleccionado}')
plt.xticks(rotation=35, ha='right')
plt.ylabel('Score')
plt.grid(axis='y', alpha=0.25)
plt.show()


## 8. Conclusión del módulo
El sistema recomienda destinos personalizados usando similitud entre usuarios. En una empresa real se puede ampliar con filtros por precio, disponibilidad, ciudad de origen, temporada y diversidad de destinos.
